In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [3]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[sql_query]
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke({
    "messages": [question]
})

In [6]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='f0fbb1be-4f26-436a-8be1-31c76a9875e1'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1303, 'prompt_tokens': 142, 'total_tokens': 1445, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1216, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D7pWhu6FAUBBbvAqb6PkapqlRVJgz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4972-06d9-7121-8686-c1a0a6aad289-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT a.name, COALESCE(SUM(t.play_count), 0) AS popularity\nFROM artists a\nLEFT JOIN tracks t ON t.artist_id = a.art

In [7]:
print(response['messages'][-1].content)

The most popular artist whose name begins with 'S' is: Smashing Pumpkins.


In [8]:
question = HumanMessage(content="Who is the most popular artist beginning with 'L' in this database?")

response = agent.invoke({
    "messages": [question]
})

print(response['messages'][-1].content)

Led Zeppelin. Based on total revenue from invoices, they have 86.13, the highest among artists whose names start with 'L'.


In [9]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT ar.Name, SUM(il.UnitPrice * il.Quantity) AS total_revenue
FROM Artist ar
JOIN Album al ON al.ArtistId = ar.ArtistId
JOIN Track t ON t.AlbumId = al.AlbumId
JOIN InvoiceLine il ON il.TrackId = t.TrackId
JOIN Invoice i ON i.InvoiceId = il.InvoiceId
WHERE LOWER(ar.Name) LIKE 'l%'
GROUP BY ar.Name
ORDER BY total_revenue DESC
LIMIT 1;
